# Example 08 — FE Beam Cubic Spring: NMA Backbone Curve

Nonlinear Modal Analysis (NMA) backbone curve and spatial mode shape for a clamped-free Euler-Bernoulli beam with a cubic spring at the midpoint, using Galerkin modal reduction to SDOF and omega-parametrised arc-length continuation. (Krack & Gross 2019, §3)

In [ ]:
# --- Imports ---
import sys
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.linalg import eigh

_repo_root = Path('..').resolve()
if str(_repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(_repo_root / 'src'))

from nlvib.systems.fe_beam import FE_EulerBernoulliBeam
from nlvib.systems.oscillators import SingleMassOscillator
from nlvib.nonlinearities.elements import cubic_spring
from nlvib.solvers.harmonic_balance import hb_residual_nma
from nlvib.continuation.solver import ContinuationSolver, ContinuationOptions

In [ ]:
# --- Full beam + mode shape extraction ---
N_ELEMENTS = 10; L_BEAM = 1.0; E_MOD = 2.1e11; I_AREA = 1e-8; RHO = 7800.0; A_SECT = 1e-4
K3_CUBIC = 1e12; CUBIC_NODE = 5; N_HARMONICS = 3

beam_full = FE_EulerBernoulliBeam(n_elements=N_ELEMENTS, L=L_BEAM, E=E_MOD,
                                   I_area=I_AREA, rho=RHO, A=A_SECT, bc='clamped-free')
K_dense = beam_full.K.toarray(); M_dense = beam_full.M.toarray()
eigvals, eigvecs = eigh(K_dense, M_dense)
omega1_linear = float(np.sqrt(np.abs(eigvals[0])))
phi1_raw = eigvecs[:, 0].real
mass_norm = float(np.sqrt(phi1_raw @ M_dense @ phi1_raw))
phi1 = phi1_raw / mass_norm

midpoint_dof = beam_full.find_dof(CUBIC_NODE, 'w')
phi1_mid = float(phi1[midpoint_dof])
tip_dof = beam_full.find_dof(N_ELEMENTS, 'w')
phi1_tip = float(phi1[tip_dof])
k3_modal = K3_CUBIC * (phi1_mid ** 4)

print(f'Full beam: n_dof={beam_full.n_dof}, omega1={omega1_linear:.2f} rad/s')
print(f'phi1 at midpoint (node {CUBIC_NODE}): {phi1_mid:.6f}')
print(f'phi1 at tip (node {N_ELEMENTS}):      {phi1_tip:.6f}')
print(f'k3_modal (projected) = {k3_modal:.4e}')

In [ ]:
# --- SDOF modal surrogate for NMA ---
sdof = SingleMassOscillator(m=1.0, d=0.0, k=omega1_linear**2)
sdof.add_nonlinear_element(cubic_spring(k3=k3_modal, dof_index=0))
n_dof_sdof = sdof.n_dof
n_total = n_dof_sdof * (2 * N_HARMONICS + 1)
print(f'SDOF NMA: n_total={n_total}, state dim={n_total+1}')

def nma_residual_omega_param(Q, omega):
    z = np.append(Q, omega)
    R_full, J_full = hb_residual_nma(z, sdof, N_HARMONICS)
    R = np.delete(R_full, n_total)
    J = J_full[:n_total, :n_total]
    return R, J

In [ ]:
# --- Initial point + continuation ---
INITIAL_MODAL_AMP = 1e-8
Q0 = np.zeros(n_total)
Q0[n_dof_sdof * 2] = INITIAL_MODAL_AMP
omega0 = omega1_linear
for _ in range(30):
    R_c, J_c = nma_residual_omega_param(Q0, omega0)
    if np.linalg.norm(R_c) < 1e-10: break
    try: dQ = np.linalg.solve(J_c, -R_c)
    except np.linalg.LinAlgError: dQ = np.linalg.lstsq(J_c, -R_c, rcond=None)[0]
    Q0 += dQ
print(f'Refined initial residual: {np.linalg.norm(R_c):.3e}')

OMEGA_MAX_BACKBONE = omega1_linear * 2.0
opts = ContinuationOptions(
    ds_initial=0.01, ds_min=1e-8, ds_max=2.0, max_steps=200,
    newton_tol=1e-8, max_newton_iter=20,
    lambda_min=omega1_linear * 0.9, lambda_max=OMEGA_MAX_BACKBONE,
)
result = ContinuationSolver().run(nma_residual_omega_param, Q0, omega0, opts)
print(f'NMA continuation: {result.n_steps} steps, {result.message}')

In [ ]:
# --- Extract backbone and save plots ---
solutions = result.solutions
omega_backbone = solutions[:, n_total]
cos1_modal = solutions[:, n_dof_sdof * 1]
sin1_modal = solutions[:, n_dof_sdof * 2]
amp_modal = np.sqrt(cos1_modal**2 + sin1_modal**2)
amp_tip = amp_modal * abs(phi1_tip)

valid_mask = (omega_backbone > 0.5 * omega1_linear) & np.isfinite(omega_backbone)
omega_bb = omega_backbone[valid_mask]
amp_bb = amp_tip[valid_mask]

if len(omega_bb) > 1:
    freq_min, freq_max = float(np.min(omega_bb)), float(np.max(omega_bb))
    amp_min, amp_max = float(np.min(amp_bb)), float(np.max(amp_bb))
else:
    freq_min = freq_max = amp_min = amp_max = float('nan')

output_dir = Path('..') / 'examples' / '08_beam_cubic_spring_nma' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

fig_bb, ax_bb = plt.subplots(figsize=(8, 5))
ax_bb.plot(amp_bb, omega_bb, color='tab:blue', linewidth=2.0,
           label=f'NMA backbone (H={N_HARMONICS}, modal reduction)')
ax_bb.axhline(omega1_linear, color='gray', linestyle='--', linewidth=0.8,
              label=f'linear omega1 = {omega1_linear:.1f} rad/s')
ax_bb.set_xlabel(r'Tip amplitude $|w_{\rm tip}|$ (m)')
ax_bb.set_ylabel(r'Natural frequency $\omega_n$ (rad/s)')
ax_bb.set_title('Example 08 — Beam Cubic Spring NMA Backbone Curve')
ax_bb.legend(); ax_bb.grid(True, alpha=0.3); fig_bb.tight_layout()
fig_bb.savefig(output_dir / 'backbone.png', dpi=150)
print('Saved backbone.png')

# Mode shape at mid-amplitude backbone point
valid_indices = np.where(valid_mask)[0]
if len(valid_indices) >= 3: mid_sol_idx = valid_indices[len(valid_indices) // 2]
elif len(valid_indices) > 0: mid_sol_idx = valid_indices[0]
else: mid_sol_idx = 0

eta_mid = float(amp_modal[mid_sol_idx])
omega_mid = float(omega_backbone[mid_sol_idx])
q_physical = eta_mid * phi1

node_positions = [0.0]; mode_disp = [0.0]
for node_i in range(1, N_ELEMENTS + 1):
    try:
        r_dof = beam_full.find_dof(node_i, 'w')
        node_positions.append(node_i * L_BEAM / N_ELEMENTS)
        mode_disp.append(float(q_physical[r_dof]))
    except ValueError: pass

node_positions_arr = np.array(node_positions); mode_disp_arr = np.array(mode_disp)
fig_mode, ax_mode = plt.subplots(figsize=(8, 4))
ax_mode.plot(node_positions_arr, np.zeros_like(node_positions_arr), 'k--', linewidth=0.6, label='undeformed')
ax_mode.plot(node_positions_arr, mode_disp_arr, 'o-', color='tab:green', linewidth=1.5,
             label=f'mode shape at omega={omega_mid:.1f} rad/s')
ax_mode.fill_between(node_positions_arr, mode_disp_arr, alpha=0.2, color='tab:green')
ax_mode.set_xlabel('Position along beam (m)'); ax_mode.set_ylabel('Transverse displacement (m)')
ax_mode.set_title('Example 08 — NMA Mode Shape at Mid-Amplitude Backbone Point')
ax_mode.legend(); ax_mode.grid(True, alpha=0.3); fig_mode.tight_layout()
fig_mode.savefig(output_dir / 'mode_shape.png', dpi=150)
print('Saved mode_shape.png')

In [ ]:
# --- Display backbone inline ---
fig_bb

In [ ]:
# --- Display mode shape inline ---
fig_mode

In [ ]:
# --- Summary ---
print('=' * 65)
print('SUMMARY — Example 08: Beam Cubic Spring NMA')
print('=' * 65)
print(f'  Linear 1st natural frequency  : {omega1_linear:.2f} rad/s')
print(f'  Backbone frequency range      : [{freq_min:.2f}, {freq_max:.2f}] rad/s')
print(f'  Backbone tip amplitude range  : [{amp_min:.3e}, {amp_max:.3e}] m')
print(f'  k3_modal (Galerkin projected) : {k3_modal:.4e}')
print(f'  Method: single-mode Galerkin reduction + omega-continuation, H={N_HARMONICS}')
print('=' * 65)